In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# Connect to database
db_path = r"C:\Users\tania\OneDrive\Documents\2026\2026 Tech Projects\sql-analytics-portfolio\food_economy_analysis\FoodPrices.db"
conn = sqlite3.connect(db_path)

# Year columns we'll use throughout
year_cols = ['Y2015','Y2016','Y2017','Y2018','Y2019',
             'Y2020','Y2021','Y2022','Y2023']

# Load PPI data
df = pd.read_sql_query("""
    SELECT Area, Item, Element,
           Y2015, Y2016, Y2017, Y2018, Y2019,
           Y2020, Y2021, Y2022, Y2023
    FROM food_prices
    WHERE Element = 'Producer Price Index (2014-2016 = 100)'
    AND Area != ''
    AND Item != ''
""", conn)

# Clean year columns
df[year_cols] = df[year_cols].replace('', None)
df[year_cols] = df[year_cols].astype(float)

print(f"✅ Loaded {len(df):,} rows")

✅ Loaded 13,268 rows


In [2]:
# Filter South Africa
sa = df[df['Area'] == 'South Africa'].copy()

# Melt to long format
sa_melted = sa.melt(
    id_vars=['Area', 'Item'],
    value_vars=year_cols,
    var_name='Year',
    value_name='PPI'
).dropna()

sa_melted['Year'] = sa_melted['Year'].str.replace('Y', '').astype(int)

print(f"South Africa has {sa['Item'].nunique()} food items tracked")
print(sa['Item'].unique())

South Africa has 118 food items tracked
<StringArray>
[                      'Apples',                     'Apricots',
                    'Asparagus',                     'Avocados',
                      'Bananas',                       'Barley',
                   'Beans, dry',                    'Buckwheat',
                     'Cabbages', 'Cantaloupes and other melons',
 ...
      'Jute & Jute-like Fibres',                    'Livestock',
                  'Meat, Total',                  'Milk, Total',
     'Oilcrops, Oil Equivalent',                'Pulses, Total',
      'Roots and Tubers, Total',              'Treenuts, Total',
 'Vegetables and Melons, Total',           'Vegetables Primary']
Length: 118, dtype: str


In [4]:
sa_staples = ['Wheat', 'Maize (corn)', 'Barley', 'Milk, Total', 
              'Meat, Total', 'Eggs, Total']

sa_filtered = sa_melted[sa_melted['Item'].isin(sa_staples)]

fig1 = px.line(
    sa_filtered,
    x='Year',
    y='PPI',
    color='Item',
    title='South Africa: Staple Food Price Trends (2015-2023)',
    labels={'PPI': 'Producer Price Index (2014-2016=100)'},
    markers=True
)
fig1.add_hline(y=100, line_dash='dash', line_color='grey',
               annotation_text='Baseline (2014-2016=100)')
fig1.update_layout(height=500)
fig1.show()

🌾 Wheat peaked at 180 in 2022 — directly linked to the Ukraine war since South Africa imports a significant amount of wheat
🌽 Maize is very volatile — dropped to 70 in 2017 (good harvest year) then spiked to 163 in 2022, this is significant since maize is a staple food for millions of South Africans
🥛 Milk shows a steady consistent rise — now 75% above baseline, quietly becoming less affordable
🥩 Meat has risen steadily since 2019 with no sign of coming down
📉 Barley is interesting — it crashed in 2021 then spiked sharply, very volatile

In [5]:
sa_fruit_veg = ['Apples', 'Bananas', 'Oranges', 'Grapes',
                'Tomatoes', 'Potatoes', 'Onions, dry',
                'Vegetables and Melons, Total']

sa_fv = sa_melted[sa_melted['Item'].isin(sa_fruit_veg)]

fig2 = px.line(
    sa_fv,
    x='Year',
    y='PPI',
    color='Item',
    title='South Africa: Fruit & Vegetable Price Trends (2015-2023)',
    labels={'PPI': 'Producer Price Index (2014-2016=100)'},
    markers=True
)
fig2.add_hline(y=100, line_dash='dash', line_color='grey',
               annotation_text='Baseline (2014-2016=100)')
fig2.update_layout(height=500)
fig2.show()

🍊 Oranges & 🥔 Potatoes both spiked sharply to 235+ in 2023 — the biggest rise of any fruit or vegetable
🍅 Tomatoes jumped dramatically in 2022-2023 — likely linked to load shedding in South Africa affecting cold storage and distribution
🍌 Bananas have been consistently rising since 2016 and hit 165 by 2023
🍇 Grapes show the most stable trend overall — still 70% above baseline but rising gradually
All items crossed well above the 100 baseline by 2023 — nothing got cheaper

In [6]:
# Find top 10 items with biggest PPI rise in South Africa
sa_change = sa.copy()
sa_change['change'] = sa_change['Y2023'] - sa_change['Y2019']
sa_change = sa_change.dropna(subset=['change'])
sa_change = sa_change.nlargest(10, 'change')

fig3 = px.bar(
    sa_change,
    x='change',
    y='Item',
    orientation='h',
    title='South Africa: Top 10 Most Inflated Food Items (2019-2023)',
    labels={'change': 'PPI Change (index points)', 'Item': 'Food Item'},
    color='change',
    color_continuous_scale='Reds'
)
fig3.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=500
)
fig3.show()

🫛 Peas, green at 1000+ points is so far ahead of everything else it's likely a data anomaly worth flagging — either a genuine extreme shortage or a data reporting issue in the FAO dataset
🎃 Pumpkins, squash and gourds at 350 points is a genuine significant rise — these are common affordable vegetables in South Africa
🥦 Cauliflowers and broccoli at 250 points is notable — these were traditionally affordable vegetables
🧅 Onions and Potatoes appearing here confirms what we saw in Cell 4 — everyday staples are becoming significantly more expensive
🍅 Tomatoes appears again — consistent with the Cell 4 finding

In [7]:
# Investigate the Peas spike
peas = df[(df['Area'] == 'South Africa') & 
          (df['Item'] == 'Peas, green')]
print(peas[year_cols].to_string())

        Y2015   Y2016   Y2017   Y2018  Y2019   Y2020   Y2021   Y2022    Y2023
11199  101.89  133.34  126.29  147.73  286.0  154.54  147.21  151.91  1343.84


What Likely Happened

Peas were already expensive in South Africa in 2019
Prices actually dropped during COVID (2020-2021) — possibly reduced demand or imports filling supply
Then in 2023 something caused a dramatic supply shock — possibly:

Severe load shedding affecting cold storage (green peas are perishable)
A bad harvest year
Export demand pulling supply away from local market



Recommendation for insights.md
This is worth flagging as a data quality note — while the spike appears genuine, a 1343 index is so extreme it warrants caution. Note it as:

"Peas, green showed an extraordinary PPI of 1343 in South Africa in 2023, representing either a genuine severe supply shock or a possible data reporting anomaly. Further verification recommended."

In [8]:
save_path = r"C:\Users\tania\OneDrive\Documents\2026\2026 Tech Projects\sql-analytics-portfolio\food_economy_analysis\visualisations\charts"

os.makedirs(save_path, exist_ok=True)

fig1.write_html(os.path.join(save_path, '04_sa_staples_trends.html'))
fig2.write_html(os.path.join(save_path, '05_sa_fruit_veg_trends.html'))
fig3.write_html(os.path.join(save_path, '06_sa_top10_inflated.html'))

print("✅ All charts saved!")

✅ All charts saved!
